In [22]:
import itertools
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [23]:
features = ['TOTAL_SPENDING', 'FREQUENCY', 'RETURN_RATIO', 'AVG_PURCHASE_GAP', 'AVG_ORDER_VALUE',"ONE_TIME_CUSTOMER","Recency"]

In [24]:
final_df = pd.read_csv("finally.csv")
final_df.to_excel("shambhawi.xlsx",index= False)

In [25]:
cols = [
    'TOTAL_SPENDING',
    'FREQUENCY',
    # 'PURCHASE_INTENSITY',
    'AVG_ORDER_VALUE',
    'AVG_PURCHASE_GAP'
]

for col in cols:
    lower = final_df[col].quantile(0.01)   
    upper = final_df[col].quantile(0.99)   

    final_df[col] = np.where(final_df[col] < lower, lower, final_df[col])
    final_df[col] = np.where(final_df[col] > upper, upper, final_df[col])

final_df["PURCHASE_INTENSIT"] = np.sqrt(final_df["PURCHASE_INTENSITY"])

In [26]:
features = [
    'TOTAL_SPENDING',
    'FREQUENCY',
    'RETURN_RATIO',
    'AVG_PURCHASE_GAP',
    'AVG_ORDER_VALUE',
    'Recency'
]

results = []

for r in range(3, len(features)+1):
    for combo in itertools.combinations(features, r):
        try:
            
            X = final_df[list(combo)].copy()
            X = X.replace([np.inf, -np.inf], 0).fillna(0)

            
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X)

            
            kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
            labels = kmeans.fit_predict(X_scaled)

            
            if len(set(labels)) > 1:
                score = silhouette_score(X_scaled, labels)

                results.append({
                    'features': combo,
                    'score': score
                })

        except Exception as e:
            print("Error:", combo, "|", e)


results_df = pd.DataFrame(results)

if not results_df.empty:
    results_df = results_df.sort_values(by='score', ascending=False)
    print(results_df.head(10))
else:
    print("No valid combinations found ")

                                             features     score
10        (FREQUENCY, RETURN_RATIO, AVG_PURCHASE_GAP)  0.549568
4    (TOTAL_SPENDING, RETURN_RATIO, AVG_PURCHASE_GAP)  0.548029
16  (RETURN_RATIO, AVG_PURCHASE_GAP, AVG_ORDER_VALUE)  0.546373
13     (FREQUENCY, AVG_PURCHASE_GAP, AVG_ORDER_VALUE)  0.532518
7   (TOTAL_SPENDING, AVG_PURCHASE_GAP, AVG_ORDER_V...  0.531940
1       (TOTAL_SPENDING, FREQUENCY, AVG_PURCHASE_GAP)  0.529371
5     (TOTAL_SPENDING, RETURN_RATIO, AVG_ORDER_VALUE)  0.526401
11         (FREQUENCY, RETURN_RATIO, AVG_ORDER_VALUE)  0.522338
20  (TOTAL_SPENDING, FREQUENCY, RETURN_RATIO, AVG_...  0.514224
3                (TOTAL_SPENDING, FREQUENCY, Recency)  0.503406


In [27]:
results_df.to_excel("feature_selection_results.xlsx", index=False)